<a href="https://colab.research.google.com/github/nataliehany/FlyRank-ML-Internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nataliehany/FlyRank-ML-Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### Method

I use a **Random Forest classifier** for the CTR / Engagement Opportunity Scoring lane.

The goal is to estimate whether a page represents a meaningful CTR review opportunity using observed search, content, and engagement signals. Random Forest is appropriate because the relationships between these signals may be nonlinear and may interact. For example, the meaning of a CTR value can depend on both search visibility and average ranking position.

The model also produces class probabilities. These probabilities can be used as an opportunity score for ranking pages rather than treating every prediction as an automatic action.

I will compare the model against the transparent Week-4 baseline rule on the same held-out split and using the same evaluation target. The model is useful only if it provides measurable discrimination beyond that simpler rule.

This remains a decision-support model. A high score means a page is worth reviewing; it does not prove that the page needs an edit or that an edit would causally increase search traffic.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np
import pandas as pd
import sklearn

from sklearn.ensemble import RandomForestClassifier

RANDOM_SEED = 42

print("Random seed:", RANDOM_SEED)
print("pandas version:", pd.__version__)
print("numpy version:", np.__version__)
print("scikit-learn version:", sklearn.__version__)
print("Model:", RandomForestClassifier.__name__)

Random seed: 42
pandas version: 2.2.3
numpy version: 2.1.3
scikit-learn version: 1.6.1
Model: RandomForestClassifier


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### Split design

I use a **grouped train/test split by anonymized client ID**.

Pages belonging to the same client may share content strategy, audience, search demand, and measurement characteristics. A random page-level split could therefore place very similar pages from the same client in both training and test data and make generalization performance look stronger than it really is.

The grouped split keeps each client entirely in either training or test. The model is trained on approximately 80% of the client groups and evaluated on the remaining 20%.

`client_id` is used only to construct the split. It is not a model feature.

The evaluation therefore asks a harder and more useful question: whether the observed relationships learned from one set of clients transfer directionally to pages from clients that were not used for model fitting.

The split is fixed with random seed 42 so the comparison is reproducible. The Week-4 baseline and the model will both be evaluated on this exact same held-out test set.

In [ ]:
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit

# Clone the internship repository if it is not already available.
REPO_PATH = Path("/content/FlyRank-ML-Internship")

if not REPO_PATH.exists():
    !git clone https://github.com/nataliehany/FlyRank-ML-Internship.git

DATA_PATH = REPO_PATH / "data/raw/content_refresh_anonymized.csv"

print("Data path exists:", DATA_PATH.exists())

df = pd.read_csv(DATA_PATH)

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Unique client groups:", df["client_id"].nunique())

Data path exists: True
Rows: 30000
Columns: 44
Unique client groups: 32


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit

DATA_PATH = Path(
    "/content/FlyRank-ML-Internship/data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(DATA_PATH)

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Unique client groups:", df["client_id"].nunique())

Rows: 30000
Columns: 44
Unique client groups: 32


In [ ]:
# Build a reproducible grouped split by anonymized client.
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_SEED
)

train_idx, test_idx = next(
    splitter.split(df, groups=df["client_id"])
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

train_clients = set(train_df["client_id"])
test_clients = set(test_df["client_id"])
client_overlap = train_clients.intersection(test_clients)

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))
print("Train clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Client overlap:", len(client_overlap))
print("Test share:", round(len(test_df) / len(df), 3))

assert len(client_overlap) == 0
print("\nGrouped split check: PASS")

Train rows: 23837
Test rows: 6163
Train clients: 25
Test clients: 7
Client overlap: 0
Test share: 0.205

Grouped split check: PASS


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

### Target and comparison

The modeling task is to identify visible pages that under-capture clicks.

I restrict the modeling frame to pages with at least 732 observed impressions and an average search position of 20 or better. Within that visible-page population, the binary target is:

- `1` = observed CTR below 0.10
- `0` = observed CTR of 0.10 or higher

The 0.10 threshold is carried forward from the Week-4 signal audit rather than selected after looking at test-set model performance.

To prevent direct target leakage, `ctr` and `clicks_90d` are used to define or inspect the outcome but are not model features. `client_id` and `content_id` are identifiers only and are also excluded.

The Random Forest will use only observed, non-label inputs. Its predicted probability for the low-CTR class becomes the model opportunity score.

For comparison, the Week-4 baseline is evaluated on the same held-out client groups and against the same binary target. Model and baseline are therefore compared on the same rows, split, and outcome.

### Features and evaluation

The model uses observed page characteristics and search/engagement context that do not directly define the target:

- `impressions_90d`
- `avg_position`
- `search_volume`
- `competition`
- `word_count`
- `content_age_days`
- `days_since_last_update`
- `pageviews_90d`
- `sessions_90d`
- `engaged_sessions_90d`
- `scroll_events_90d`

I deliberately exclude `ctr` and `clicks_90d` because CTR defines the target and clicks are a direct component of CTR. I also exclude identifiers, derived trend fields, tier fields, and categorical product-generated labels.

The test positive rate is 22.3%, while the majority-class base rate is 77.7%. Accuracy alone would therefore be misleading.

I compare the Random Forest and Week-4 baseline primarily using ROC-AUC, which measures discrimination across thresholds. I also report precision, recall, and F1 at the chosen classification threshold for context.

### Features and evaluation

The model uses observed page characteristics and search/engagement context that do not directly define the target.

I deliberately exclude `ctr` and `clicks_90d` because CTR defines the target and clicks are a direct component of CTR. I also exclude identifiers from the model features.

The test positive rate is 22.3%, while the majority-class accuracy base rate is 77.7%. Accuracy alone would therefore be misleading.

The Random Forest and Week-4 baseline are compared primarily using ROC-AUC on the same held-out client groups. ROC-AUC evaluates their ability to rank the observed low-CTR cases above the other cases without requiring the same classification threshold.

For the Random Forest, I also report accuracy, precision, recall, and F1 at the fixed 0.50 classification threshold for context. These thresholded metrics are not used for the baseline comparison.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Create the visible-page modeling population.
def make_model_frame(frame):
    out = frame[
        (frame["impressions_90d"] >= 732) &
        (frame["avg_position"] <= 20) &
        frame["ctr"].notna()
    ].copy()

    out["low_ctr_target"] = (out["ctr"] < 0.10).astype(int)
    return out


train_model = make_model_frame(train_df)
test_model = make_model_frame(test_df)

print("Train modeling rows:", len(train_model))
print("Test modeling rows:", len(test_model))

print("\nTRAIN TARGET")
print(train_model["low_ctr_target"].value_counts())
print(
    "Positive rate:",
    round(train_model["low_ctr_target"].mean(), 3)
)

print("\nTEST TARGET")
print(test_model["low_ctr_target"].value_counts())
print(
    "Positive rate:",
    round(test_model["low_ctr_target"].mean(), 3)
)

print(
    "\nMajority-class test base rate:",
    round(test_model["low_ctr_target"].value_counts(normalize=True).max(), 3)
)

Train modeling rows: 8925
Test modeling rows: 2005

TRAIN TARGET
low_ctr_target
0    6944
1    1981
Name: count, dtype: int64
Positive rate: 0.222

TEST TARGET
low_ctr_target
0    1558
1     447
Name: count, dtype: int64
Positive rate: 0.223

Majority-class test base rate: 0.777


In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

FEATURES = [
    "impressions_90d",
    "avg_position",
    "search_volume",
    "competition",
    "word_count",
    "content_age_days",
    "days_since_last_update",
    "pageviews_90d",
    "sessions_90d",
    "engaged_sessions_90d",
    "scroll_events_90d"
]

X_train = train_model[FEATURES]
y_train = train_model["low_ctr_target"]

X_test = test_model[FEATURES]
y_test = test_model["low_ctr_target"]

# Random Forest model
model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("rf", RandomForestClassifier(
        n_estimators=300,
        random_state=RANDOM_SEED,
        n_jobs=-1,
        class_weight="balanced",
        min_samples_leaf=5
    ))
])

model.fit(X_train, y_train)

# Random Forest probability score and fixed 0.50 prediction threshold
model_score = model.predict_proba(X_test)[:, 1]
model_pred = (model_score >= 0.50).astype(int)

# Week-4 transparent baseline score.
# The modeling population is already restricted to:
# impressions_90d >= 732 and avg_position <= 20.
baseline_score = (
    np.log1p(test_model["impressions_90d"])
    + np.where(test_model["avg_position"] <= 10, 1.0, 0.0)
)

# Fair model-vs-baseline comparison:
# compare continuous ranking/discrimination using ROC-AUC.
comparison = pd.DataFrame({
    "Method": ["Week-4 baseline", "Random Forest"],
    "ROC_AUC": [
        roc_auc_score(y_test, baseline_score),
        roc_auc_score(y_test, model_score)
    ]
})

# Threshold-dependent metrics are reported only for Random Forest.
rf_threshold_metrics = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1"
    ],
    "Random_Forest_at_0.50": [
        accuracy_score(y_test, model_pred),
        precision_score(y_test, model_pred, zero_division=0),
        recall_score(y_test, model_pred, zero_division=0),
        f1_score(y_test, model_pred, zero_division=0)
    ]
})

print("Test rows:", len(y_test))
print("Positive rate:", round(y_test.mean(), 3))
print(
    "Majority-class accuracy base rate:",
    round(y_test.value_counts(normalize=True).max(), 3)
)

print("\nMODEL VS BASELINE — SAME HELD-OUT SPLIT")
display(comparison.round(3))

print("\nRANDOM FOREST — 0.50 THRESHOLD")
display(rf_threshold_metrics.round(3))

Test rows: 2005
Positive rate: 0.223
Majority-class accuracy base rate: 0.777

MODEL VS BASELINE — SAME HELD-OUT SPLIT


,Method,ROC_AUC
0,Week-4 baseline,0.400
1,Random Forest,0.806



RANDOM FOREST — 0.50 THRESHOLD


,Metric,Random_Forest_at_0.50
0,Accuracy,0.693
1,Precision,0.402
2,Recall,0.767
3,F1,0.527


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

### Error analysis

On the held-out client groups, the Random Forest achieved ROC-AUC 0.806 compared with 0.400 for the Week-4 baseline. This indicates substantially stronger discrimination between the two observed CTR groups on this split.

At the 0.50 classification threshold, the Random Forest achieved precision 0.402, recall 0.767, and F1 0.527. Its accuracy was 0.693, which is below the 0.777 majority-class base rate. Therefore, accuracy is not evidence of improvement here. The model's useful result is its ranking/discrimination ability and relatively high recall.

The recall of 0.767 means the model identifies many of the measured low-CTR opportunities, but precision of 0.402 means many flagged pages are false positives. This is acceptable only as a decision-support queue where a human reviews candidates before taking action.

I also inspect false positives and false negatives rather than relying only on aggregate metrics. False positives represent editorial review that may not be necessary, while false negatives represent observed low-CTR pages that the model failed to prioritize.

### Interpretation

Feature importance is used as a descriptive diagnostic of what the fitted Random Forest relies on. It does not establish that a feature causes CTR performance.

The interpretation is therefore directional: the model shows whether observed page, search, and engagement characteristics contain useful information for ranking CTR-review opportunities on held-out clients.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.metrics import confusion_matrix

# Error types on the held-out test set.
error_frame = test_model[
    ["content_id", "impressions_90d", "avg_position", "ctr"]
].copy()

error_frame["actual"] = y_test.to_numpy()
error_frame["predicted"] = model_pred
error_frame["model_score"] = model_score

error_frame["error_type"] = np.select(
    [
        (error_frame["actual"] == 1) & (error_frame["predicted"] == 1),
        (error_frame["actual"] == 0) & (error_frame["predicted"] == 0),
        (error_frame["actual"] == 0) & (error_frame["predicted"] == 1),
        (error_frame["actual"] == 1) & (error_frame["predicted"] == 0)
    ],
    [
        "true_positive",
        "true_negative",
        "false_positive",
        "false_negative"
    ],
    default="unknown"
)

print("CONFUSION MATRIX")
cm = confusion_matrix(y_test, model_pred)
display(
    pd.DataFrame(
        cm,
        index=["Actual 0", "Actual 1"],
        columns=["Predicted 0", "Predicted 1"]
    )
)

print("\nERROR COUNTS")
display(
    error_frame["error_type"]
    .value_counts()
    .rename_axis("error_type")
    .reset_index(name="n")
)

# Random Forest feature importance.
rf = model.named_steps["rf"]

importance = (
    pd.DataFrame({
        "feature": FEATURES,
        "importance": rf.feature_importances_
    })
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

print("\nFEATURE IMPORTANCE")
display(importance)

print("\nHIGH-CONFIDENCE FALSE POSITIVES")
display(
    error_frame[error_frame["error_type"] == "false_positive"]
    .sort_values("model_score", ascending=False)
    .head(5)
)

print("\nHIGH-CONFIDENCE FALSE NEGATIVES")
display(
    error_frame[error_frame["error_type"] == "false_negative"]
    .sort_values("model_score", ascending=True)
    .head(5)
)

CONFUSION MATRIX


,Predicted 0,Predicted 1
Actual 0,1047,511
Actual 1,104,343



ERROR COUNTS


,error_type,n
0,true_negative,1047
1,false_positive,511
2,true_positive,343
3,false_negative,104



FEATURE IMPORTANCE


,feature,importance
0,pageviews_90d,0.150027
1,avg_position,0.134723
2,impressions_90d,0.134283
3,content_age_days,0.117318
4,sessions_90d,0.117225
5,engaged_sessions_90d,0.091651
6,word_count,0.085237
7,scroll_events_90d,0.061042
8,competition,0.040477
9,search_volume,0.039411



HIGH-CONFIDENCE FALSE POSITIVES


,content_id,impressions_90d,avg_position,ctr,actual,predicted,model_score,error_type
23763,content_fd15559290a1,1154,8.2,0.35,0,1,0.938122,false_positive
27610,content_b2b7178dc27b,2982,8.0,0.10,0,1,0.924448,false_positive
22805,content_c79bfcc07824,1321,9.4,0.23,0,1,0.905696,false_positive
2735,content_6a9cd9678ddf,2100,17.9,0.14,0,1,0.889531,false_positive
16203,content_60bd44a91d5c,1860,14.8,0.11,0,1,0.888386,false_positive



HIGH-CONFIDENCE FALSE NEGATIVES


,content_id,impressions_90d,avg_position,ctr,actual,predicted,model_score,error_type
20891,content_6cc2663a00ec,3273,2.0,0.09,1,0,0.126988,false_negative
15618,content_a4dc39c5f8f9,2901,6.3,0.03,1,0,0.133834,false_negative
1389,content_3023680512a5,1280,4.8,0.08,1,0,0.136315,false_negative
4098,content_9467ff7ac54c,1279,4.8,0.08,1,0,0.143277,false_negative
6903,content_c84a0ab98e90,223271,7.8,0.03,1,0,0.162392,false_negative


In [ ]:
import json
from pathlib import Path

metrics = {
    "random_seed": RANDOM_SEED,
    "test_rows": int(len(y_test)),
    "test_positive_rate": float(y_test.mean()),
    "majority_class_base_rate": float(
        y_test.value_counts(normalize=True).max()
    ),
    "baseline_roc_auc": float(
        roc_auc_score(y_test, baseline_score)
    ),
    "random_forest_roc_auc": float(
        roc_auc_score(y_test, model_score)
    ),
    "random_forest_accuracy": float(
        accuracy_score(y_test, model_pred)
    ),
    "random_forest_precision": float(
        precision_score(y_test, model_pred)
    ),
    "random_forest_recall": float(
        recall_score(y_test, model_pred)
    ),
    "random_forest_f1": float(
        f1_score(y_test, model_pred)
    ),
    "true_negative": int(cm[0, 0]),
    "false_positive": int(cm[0, 1]),
    "false_negative": int(cm[1, 0]),
    "true_positive": int(cm[1, 1])
}

output_dir = Path(
    "/content/FlyRank-ML-Internship/work/outputs"
)
output_dir.mkdir(parents=True, exist_ok=True)

metrics_path = output_dir / "w05_model_metrics.json"

with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

print("Metrics written to:", metrics_path)
print(json.dumps(metrics, indent=2))

Metrics written to: /content/FlyRank-ML-Internship/work/outputs/w05_model_metrics.json
{
  "random_seed": 42,
  "test_rows": 2005,
  "test_positive_rate": 0.2229426433915212,
  "majority_class_base_rate": 0.7770573566084789,
  "baseline_roc_auc": 0.39967850137703076,
  "random_forest_roc_auc": 0.8057611289641685,
  "random_forest_accuracy": 0.6932668329177057,
  "random_forest_precision": 0.4016393442622951,
  "random_forest_recall": 0.767337807606264,
  "random_forest_f1": 0.5272867025365103,
  "true_negative": 1047,
  "false_positive": 511,
  "false_negative": 104,
  "true_positive": 343
}


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.